In [1]:
import numpy as np
import pandas as pd
import os, os.path,sys
from sklearn.preprocessing import MinMaxScaler
import tqdm
from tensorflow import keras
import tensorflow as tf
from keras import backend as K
from keras import layers, Input
from keras.models import Sequential, Model
from keras.optimizers import Adam

import matplotlib.pyplot as plt

2025-01-24 15:08:00.246486: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-24 15:08:00.246720: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-24 15:08:00.248281: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-24 15:08:00.423571: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
def make_model_lstm_block2(lookback):
        inputs = Input((lookback, 1))
        model = layers.LSTM(128)(inputs)
        #model = layers.Dense(64)(model)
        model = layers.Dropout(0.3)(model, training=True) ### original 0.2
        output = layers.Dense(1)(model)
        model = Model(inputs, output)
        return model

In [ ]:
 def cost_func(y_true, y_pred, alpha, delay, gamma):
            step = alpha
            forecast_delay = delay
            #delta_x = 0.05
            epsilon = -step / 0.1
            y_pred = tf.expand_dims(y_pred, axis=1)
            y_pred = tf.tile(y_pred, (1, forecast_delay, 1))
            cost = y_pred - y_true# - delta_x
            big_penalty = epsilon * (cost + 0.1) + step
            penalty = -0.1 * (cost + 0.1) + step
            pen_positive = cost * gamma
            cost = tf.where(cost > 0, pen_positive, cost)
            cost = tf.where(cost < -0.1, penalty, cost)
            cost = tf.where(tf.logical_and((cost <= 0), (cost >= -0.1)),
                            big_penalty, cost)
            cost = tf.abs(cost)
            cost = K.sum(K.sum(cost, axis=-1), axis=-1)
            cost= K.mean(cost) ## ADDED NOW, Maybe remove it
            return cost

In [ ]:

LOOKBACK = 6 # History given as input to the network. Could be modified if needed
GAMMA = 2 # Positive slope of the loss function
ALPHA=5 # Negative slope of the loss function
B = 100 # Number of montecarlo output
batch_size = 128

In [ ]:

###HERE YOU HAVE TO LOAD YOUR DATA ######

bordeaux = pd.read_csv(f'/home/sergi_alcala/sergi_data/AZTEC_extension/citys/{city}.csv')
bordeaux.drop('date_time', axis=1, inplace=True)
bordeaux = bordeaux.reindex(sorted(bordeaux.columns), axis=1)
bordeaux = bordeaux.to_numpy()

####################

XTRAIN=round(len(bordeaux)*0.8) # 80% of the data is used for training

## Normalize Data
minmaxscaler = MinMaxScaler()
x_train = bordeaux[:XTRAIN]
x_train_norm = tf.convert_to_tensor(minmaxscaler.fit_transform(x_train), dtype=tf.float32)
x_test = bordeaux[XTRAIN:]
x_test_norm = tf.convert_to_tensor(minmaxscaler.transform(x_test), dtype=tf.float32)

print(f'lenght of x_train_norm: {len(x_train_norm)}, lenght of x_test_norm: {len(x_test_norm)}')

In [ ]:
input_dataset = keras.preprocessing.timeseries_dataset_from_array(x_train_norm[:-DELAY], None,
                                                                    sequence_length=LOOKBACK, sequence_stride=DELAY)
target_dataset = keras.preprocessing.timeseries_dataset_from_array(x_train_norm[LOOKBACK:], None,
                                                                    sequence_length=DELAY, sequence_stride=DELAY)


test_dataset = keras.preprocessing.timeseries_dataset_from_array(x_test_norm[:-DELAY], None,
                                                                    sequence_length=LOOKBACK, sequence_stride=DELAY,
                                                                    batch_size=batch_size)

model = NN.make_model_lstm_block2(LOOKBACK)

optimizer = Adam(learning_rate=0.0005)


print(f' City: {city} - Phi: {PHI} - Alpha: {ALPHA} - Delay: {DELAY} - Gamma: {GAMMA} - Lookback: {LOOKBACK}')
for epoch in tqdm(range(EPOCHS)):
    #print("\nStart of epoch %d" % (epoch,))
    for step, (x_batch_train, y_batch_train) in enumerate(zip(input_dataset, target_dataset)):
        with tf.GradientTape() as tape:
            prediction = model(x_batch_train, training=True)
            loss_value = cost_func(y_batch_train, prediction, ALPHA, DELAY, GAMMA)
            
        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

load_forecasted = np.zeros((int(agg_shared_capacity_test.shape[0]/DELAY), B))

for idx, inputs in enumerate(test_dataset):
    for i in range(B):
        if inputs.shape[0] == batch_size:
            load_forecasted[idx * batch_size: (idx+1)*batch_size, i] = model.predict(inputs,verbose=0)[:, 0]
        else:
            load_forecasted[-inputs.shape[0]:, i] = model.predict(inputs,verbose=0)[:, 0]



    
load_forecasted = np.repeat(load_forecasted, DELAY, axis=0)